Build a nano GPT Part-02
-----
[video](https://youtu.be/kCc8FmEb1nY?t=2533)

<ins>Goal</ins>

Questions:
1. Why is that we are averaging past context for self attention

Self-Attention Intro
------

Self-attention is a mechanism used in neural networks, particularly in transformers, that allows the model to weigh the importance of different tokens (which can be words or characters) in the input sequence. Instead of processing tokens in isolation, self-attention enables the model to consider the context provided by all other tokens in the sequence simultaneously. This is really useful for things like understanding language and even predicting the next character in a text.

**Key Points of Self-Attention:**

*   **Contextual Understanding:** It helps the model understand the meaning of a token based on its relationship with other tokens in the same sequence.
*   **Parallel Processing:** Unlike traditional sequential models (like RNNs), self-attention can process all tokens in a sequence in parallel, making it more efficient for long sequences.
*   **Variable Dependencies:** It can capture long-range dependencies between tokens, regardless of their distance in the sequence.
*   **Learned Weights:** The importance given to each token is learned during the training process.
*   **Core of Transformers:** Self-attention is a fundamental component of the Transformer architecture, which has achieved state-of-the-art results in various NLP tasks, including character-level language modeling.

**What is Bag-of-Words?**

Bag-of-words is a much simpler way to represent text. It just counts how many times each word appears in a document. It completely ignores the order of the words and the relationships between them.

For example, the sentences "The quick brown fox jumps over the lazy dog" and "The lazy dog jumps over the quick brown fox" would have the same bag-of-words representation because they use the same words the same number of times, even though their meaning is different.

**Is Bag-of-Words used in self-attention?**

No, bag-of-words is **not** used in self-attention. Self-attention, as explained above, specifically *does* consider the relationships and order of tokens in the sequence through the Query, Key, and Value mechanism. This is a key difference that makes self-attention (and models like transformers that use it) much more powerful for understanding context and generating coherent text compared to models based on bag-of-words.


**Simplified working of self-attention**

let's use the sentence "The dog ate the bone because it was hungry." to explain the point-by-point breakdown of self-attention. In this sentence, "it" refers to "dog".

1.  **Looking at Each Token:** The computer looks at each token: "The", "dog", "ate", "the", "bone", "because", "it", "was", "hungry", "."

2.  **Asking Questions (Queries), Getting Information (Keys), and Having the Content (Values):** For *each* of these ten tokens, the computer creates a Query, a Key, and a Value. Think of them like this for the token "dog":
    *   **Query ("dog"):** "What information about other tokens is relevant to understanding 'dog'?"
    *   **Key ("dog"):** "Here is the information I can offer about 'dog'."
    *   **Value ("dog"):** The actual meaning or representation of the token "dog".

    Let's look at the token "it" as another example:
    *   **Query ("it"):** "What information about other tokens is relevant to understanding 'it'?"
    *   **Key ("it"):** "Here is the information I can offer about 'it'."
    *   **Value ("it"):** The actual meaning or representation of the token "it".

3.  **Comparing and Scoring:** This is where the magic happens. The computer takes the Query of *one* token (let's say, the Query for "it") and compares it to the Key of *every other token* in the sentence.
    *   Query ("it") vs. Key ("The") -> Score 1
    *   Query ("it") vs. Key ("dog") -> Score 2 (This score will be high!)
    *   Query ("it") vs. Key ("ate") -> Score 3
    *   Query ("it") vs. Key ("the") -> Score 4
    *   Query ("it") vs. Key ("bone") -> Score 5
    *   Query ("it") vs. Key ("because") -> Score 6
    *   Query ("it") vs. Key ("it") -> Score 7 (This score will also be high!)
    *   Query ("it") vs. Key ("was") -> Score 8
    *   Query ("it") vs. Key ("hungry") -> Score 9
    *   Query ("it") vs. Key (".") -> Score 10

    This comparison is done for the Query of *every* token against the Key of *every* other token.

4.  **Weighting the Importance:** The scores from the comparison step are converted into weights. For "it", the weight associated with "dog" will be significantly higher than the weights for most other words because the comparison score was high.

5.  **Combining Information:** To create the new, context-aware representation for "it", the computer takes the Value of *every* token in the sentence and adds them up, but each Value is multiplied by the attention weight calculated in the previous step. Since the weight for "dog" was high when considering "it", the Value of "dog" will contribute a lot to the final representation of "it". This is how the model learns that "it" in this sentence refers to "dog".

This same process of comparing Queries and Keys, getting scores, converting to weights, and combining weighted Values is done for every single token in the input sequence simultaneously. This allows the model to build a rich understanding of how all the tokens relate to each other within their context.

**The mathematical trick in self-attention**

The trick here is to find the weighted average of previous characters and there by showing us a very basic mechanism for how a model could incorporate information from the past(how much influence each past character has).

Note: The weighted average of previous characters is a simplified way to understand a core idea behind self-attention, but self-attention itself is more sophisticated.

In [58]:
import torch
import torch.nn as nn
from torch.nn import functional as F
# consider the following toy example:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [44]:
# option 1: finding weighted average/mean of past chars - using for loop
xbow = torch.zeros(B,T,C)
for b in range(B):
  for t in range(T):
    xprev = x[b,:t+1]
    xbow[b,t] = torch.mean(xprev, 0)
print(x[0])
print(xbow[0])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [56]:
# option 2: finding weighted average of past chars - using matrix multiplication
wei = torch.tril(torch.ones(T,T)) # creates a lower triangular matrix of ones
sum = wei.sum(1, keepdim=True) # calculates the sum of each row
wei = wei / wei.sum(1, keepdim=True) # normalizes the wei matrix by dividing each element by its row sum

# calculates the weighted average of past characters for each position in the sequence across all batches
xbow2 = wei @ x # here wei.shape is [T,T] and x.shape is [B,T,C] , so during matrix multiplication wei will be broadcasted to [B,T,T]
torch.allclose(xbow, xbow2) # ideally this should be true, but might show false due to differences in floating-point precision

False

In [59]:
# option 3: finding weighted average of past chars - using softmax
tril = torch.tril(torch.ones(T, T))# creates a lower triangular matrix of ones
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # masked_fill tells set inf to all elements in tril where its value is 0
wei = F.softmax(wei, dim=-1) # softmax exponentiate each element and divide by sum
xbow3 = wei @ x
torch.allclose(xbow, xbow3)# ideally this should be true, but might show false due to differences in floating-point precision

False

Initial Setup
----

In [ ]:
# download the sample dataset(tiny shakespeare dataset)
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -o input.txt -s

In [ ]:
# read the input file
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()
print("total chars in text: ", len(text))

total chars in text:  1115394


In [ ]:
# construct vocab
chars = sorted (list(set(text))) # extract all unique chars from entire text
vocab_size = len(chars)
print('all chars', chars)
print(f'{vocab_size=}')

all chars ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
vocab_size=65


In [ ]:
# construct encoder and decoder
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[ch] for ch in s] #encoder: take a string, output a list of integers
decode = lambda ix: ''.join([itos[i] for i in ix]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# tokenization/encode the entire text to integer ids
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [ ]:
## Build training & validation split from encoded data ##
n = int(0.9 * len(data)) # first 90% will be used for train and remaining 10% for validation
train_data = data[:n]
val_data = data[n:]

In [ ]:
# optional - sample showing how we construct chunks of data that we later use for training
ctx_len = 8 #context length: how many characters do we take to predict the next one?
x = train_data[:ctx_len]
y = train_data[1:ctx_len+1]
for t in range(ctx_len):
  ctx = x[:t+1]
  target = y[t]
  print(f'{ctx=}, {target=}')


ctx=tensor([18]), target=tensor(47)
ctx=tensor([18, 47]), target=tensor(56)
ctx=tensor([18, 47, 56]), target=tensor(57)
ctx=tensor([18, 47, 56, 57]), target=tensor(58)
ctx=tensor([18, 47, 56, 57, 58]), target=tensor(1)
ctx=tensor([18, 47, 56, 57, 58,  1]), target=tensor(15)
ctx=tensor([18, 47, 56, 57, 58,  1, 15]), target=tensor(47)
ctx=tensor([18, 47, 56, 57, 58,  1, 15, 47]), target=tensor(58)


In [ ]:
torch.manual_seed(1337) # ensures that the random number generation is reproducible,
ctx_len = 8 # context length: how many characters do we take to predict the next one
batch_size = 4 # how many independent sequences will we process in parallel

def get_batch(split): # generates batch(batch_size) of inputs and target data. Each item in batch is of size ctx_len
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - ctx_len, (batch_size,)) # generates batch_size (4) random integers.
  # generate input and output sequence based on above 4 random integers
  x = torch.stack([data[i:i+ctx_len] for i in ix]) # rows of input sequence
  y = torch.stack([data[i+1:i+ctx_len+1] for i in ix]) # rows of output sequence
  return x, y

xb, yb = get_batch('train') # get a batch of input and target
print(f'{xb=}')
print(f'{yb=}')

# just showing the chunks/grouping generated for the 1st item in batch
for t in range(ctx_len):
  ctx = xb[0, :t+1]
  target = yb[0, t]
  print(f"when input is {ctx.tolist()}{[itos[int(i)] for i in ctx]} the target: {target} {itos[int(target)]}")

xb=tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
yb=tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24]['L'] the target: 43 e
when input is [24, 43]['L', 'e'] the target: 58 t
when input is [24, 43, 58]['L', 'e', 't'] the target: 5 '
when input is [24, 43, 58, 5]['L', 'e', 't', "'"] the target: 57 s
when input is [24, 43, 58, 5, 57]['L', 'e', 't', "'", 's'] the target: 1  
when input is [24, 43, 58, 5, 57, 1]['L', 'e', 't', "'", 's', ' '] the target: 46 h
when input is [24, 43, 58, 5, 57, 1, 46]['L', 'e', 't', "'", 's', ' ', 'h'] the target: 43 e
when input is [24, 43, 58, 5, 57, 1, 46, 43]['L', 'e', 't', "'", 's', ' ', 'h', 'e'] the target: 39 a


Pytorch Bigram model
------

Here we are only trying to predict the next char based on the last char in input sequence and not on the entire input context

In [ ]:
# Bigram Model Definition
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # creates a lookup table that maps each input character to a vector of scores/weights (logits) representing how likely each character is to follow it.
        # The scores are randomly initialized at first and are learned during training.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # shape of input idx=[4,8] and target=[4,8]
        # lookup against the input to get the corresponding logits
        logits = self.token_embedding_table(idx) # (B-Batch,T-Times,C-Channel)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # reshaping to conform with pytorch cross_entropy. i.e [4,8,65] to [32,65]
            targets = targets.view(B*T) # reshaping to conform with pytorch cross_entropy. i.e [4,8] to [32]
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx) # calls 'forward' method to get the predictions. 'loss' can be ignored
            # focus only on the last time(T) step. -1 extracts the last element along the specified dimension(here its time dim 'T')
            logits = logits[:, -1, :] #  becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            # concatenates the newly sampled token IDs (idx_next) to the end of the current sequence (idx) along the time dimension (dim=1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1) . T dimension size increase over the iteration.
        return idx

# Create a Bigram model
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# idx = torch.zeros((1, 1), dtype=torch.long) # basically, 0 is char '\n'. we start with
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
['\n', 'S', 'r', '?', 'q', 'P', '-', 'Q', 'W', 'k', 't', 'X', 'o', 'L', '&', 'j', 'L', 'D', 'J', 'g', 'O', 'L', 'V', 'z', "'", 'R', 'I', 'o', 'D', 'q', 'H', 'd', 'h', 's', 'V', '&', 'v', 'L', 'L', 'x', 'a', 't', 'j', 's', 'c', 'M', 'p', 'w', 'L', 'E', 'R', 'S', 'P', 'y', 'a', 'o', '.', 'q', 'f', 'z', 's', '$', 'Y', 's', '$', 'z', 'F', '-', 'w', ',', ';', 'e', 'E', 'k', 'z', 'x', 'j', 'g', 'C', 'K', 'F', 'C', 'h', 's', '!', 'i', 'W', 'W', '.', 'O', 'b', 'z', 'D', 'n', 'x', 'A', ' ', 'M', 's', '$', '3']


Train pytorch Bigram model
------

In [ ]:
# create a PyTorch optimizer - here we using AdamW instead of SGD optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) # learning rate

In [ ]:
batch_size = 32 # batch size increas from 8 to 32
for steps in range(1000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb) # evaluate the loss
    optimizer.zero_grad(set_to_none=True) # zero grad
    loss.backward() # backpass
    optimizer.step() # update parameters

print(loss.item())

2.799363613128662


In [ ]:
# lets see what model can generate after training
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


humIk:dadDGowate gFouivvian mpuE!

t cl,jwnthyORS?;UGJ.PToXEbra!usqbrd zkt t m;
PURL:.
Pl?,
uf boha-veatoyayTqRGHkRao
Who ndady'zeZABLE:CEDWM:CFold CUR
l?YQ?BOviths'd V, qp taHUNUL rwain UqINbbl.Thest.ch esI'aQKwim yxas isCEGlloneve Rwonhise tYzheviq-waves cLTunghS$weatoe a
GJsalp fer toitolis!
TEN r' EaovkOK!wEHor!
GLEErix;
Slt we d&y q-u,IE mIO's&
ONw'Yptpo am fu wito ss w,
pjevitLUxfa-oo INCHes,ORGJXET:bH;
F tINGLI'giqs hauf qapUJW.
SPRIRyCigicq-gEdemIBiP :3Fbo shiDo t,
Mk-d dk, Id'
adc'sp?YZ
